# Task 1 - asking a vision model to describe an image

The model is gemma3:4b running in Ollama. Ollama has to be running for this notebook.

I wanted to use llama3.2-vision as the assignment says, but Ollama 0.32.14 cannot load it
(`unknown model architecture: 'mllama'`), so I used gemma3:4b, which also reads images.


In [1]:
import sys
sys.path.append('..')

import pandas as pd
pd.set_option('display.width', 120)


In [2]:
from src import llm
from config import setting
import json

print('ollama up:', llm.server_is_up())
print('vision model:', setting.vision_model)


ollama up: True
vision model: gemma3:4b


## The structured prompt

The prompt tells the model to describe only what it can see, not to diagnose, and to answer
with a JSON object with four fields. It is in prompts/vlm_structured.txt.


In [3]:
print(llm.load_prompt('vlm_structured'))


You are looking at one microscopy image. Describe only what is visually present.
Do not diagnose, do not name a disease, do not guess at a patient.

Reply with a single JSON object and nothing else:

{
  "modality": "the imaging technique, e.g. fluorescence microscopy, histology, x-ray",
  "tissue_type": "what the sample appears to be, or \"uncertain\"",
  "notable_features": ["short phrases describing shape, spacing, brightness, texture"],
  "image_quality": "good | acceptable | poor, followed by a short reason"
}

Rules:
- if a field cannot be judged from the image, write "uncertain" rather than guessing
- notable_features must hold between 2 and 5 entries
- no text before or after the JSON, no code fences


In [4]:
out = llm.describe_image('test_000', 'test')
print(f"{out['seconds']}s, missing keys: {out['missing_keys'] or 'none'}")
print(json.dumps(out['record'], indent=2))
llm.save_json(out, 'task1_structured_test_000')


2.5s, missing keys: none
{
  "modality": "fluorescence microscopy",
  "tissue_type": "uncertain",
  "notable_features": [
    "multiple circular fluorescent spots",
    "varying sizes of spots",
    "spots scattered across the image",
    "some spots are elongated"
  ],
  "image_quality": "acceptable",
  "reason": "image is somewhat grainy"
}


PosixPath('/Users/hoangdat/Data/Herts/Analysis_AI/vision_assignment/outputs/json/task1_structured_test_000.json')

The model got the modality right and said `uncertain` for the tissue type instead of
guessing, which is what the prompt asked for. It also added a field called `reason` that
nobody asked for, so the schema is followed but not exactly.


In [5]:
extra = [k for k in out['record'] if k not in setting.vision_keys]
print('fields I asked for that are missing:', out['missing_keys'] or 'none')
print('fields the model added by itself:', extra or 'none')


fields I asked for that are missing: none
fields the model added by itself: ['reason']


## The naive prompt, for comparison

prompts/vlm_naive.txt just asks "What do you see in this medical image?". The reply is
prose, so nothing later in the pipeline can read it.


In [6]:
llm.compare_prompts('test_000', 'test')


,prompt,chars,valid_json,keys,reply
0,vlm_naive,502,False,0,"Based on the image, I see several bright, circ..."
1,vlm_structured,293,True,5,"```json { ""modality"": ""fluorescence microsco..."


## Does the same question give the same answer twice

With temperature 0 it does. With temperature 0.7 the four fields stay the same and only the
wording of notable_features moves. So the free text wanders and the labels do not.


In [7]:
a = llm.describe_image('test_000', 'test', temperature=0.0)['record']
b = llm.describe_image('test_000', 'test', temperature=0.0)['record']
print('temperature 0, two runs identical:', a == b)


temperature 0, two runs identical: True


In [8]:
runs = llm.repeat_run('test_000', 'test', n=3, temperature=0.7)
pd.DataFrame([{'modality': r['record'].get('modality'),
               'tissue_type': r['record'].get('tissue_type'),
               'image_quality': r['record'].get('image_quality'),
               'features': '; '.join(r['record'].get('notable_features', []))} for r in runs])


,modality,tissue_type,image_quality,features
0,fluorescence microscopy,uncertain,acceptable,multiple circular fluorescent spots; varying s...
1,fluorescence microscopy,uncertain,acceptable,multiple circular fluorescent spots; varying s...
2,fluorescence microscopy,uncertain,good,multiple circular fluorescent spots; varying s...


## The corrupted images

This is the interesting one. The blurred image and the contrast crushed image are obviously
damaged, but the model does not say so. The low contrast one even gets a better quality
rating than the clean original.


In [9]:
corrupt = llm.describe_corrupted()
corrupt.to_csv('../outputs/csv/task1_vlm_corrupted.csv', index=False)
corrupt


,base_id,variant,modality,image_quality,features
0,test_000,clean,fluorescence microscopy,acceptable,multiple circular fluorescent spots; varying s...
1,test_000,blur,microscopy,acceptable,multiple small blue circles; varying brightnes...
2,test_000,clean,fluorescence microscopy,acceptable,multiple circular fluorescent spots; varying s...
3,test_000,lowcontrast,microscopy,good,multiple small circles; varying brightness; ra...
4,test_004,clean,fluorescence microscopy,good,"blue, circular shapes; varying sizes; irregula..."
5,test_004,blur,microscopy,acceptable,blue dots; irregular shapes; varying brightness
6,test_004,clean,fluorescence microscopy,good,"blue, circular shapes; varying sizes; irregula..."
7,test_004,lowcontrast,microscopy,good,multiple circular shapes; dark gray background...


So the vision model can tell you what kind of image it is, but it never gives a count and
it does not notice when the image is broken. Task 2 goes the other way round: the model
gets numbers and never sees the image.
